# Phase 6.1: Environment Initialization and Data Flash-Extraction
Mounts Google Drive to get checkpoints and extracts the raw dataset zip file to local Colab storage.

In [1]:
import os
import shutil
from google.colab import drive

# Mount Google Drive for persistent state tracking storage
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# Define raw archive path and fast local runtime target destination paths
# DATASET_ZIP = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/data/BraTS2020_TrainingData.zip"
# DATASET_ZIP = "/content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_128.zip"
# DATASET_ZIP = "/content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_128_Cubic_B_spline.zip"
DATASET_ZIP = "/content/drive/MyDrive/ML-Datasets/BraTS2020_TrainingData_ABC.zip"
LOCAL_EXTRACT_DIR = "/content/MICCAI_BraTS2020_TrainingData_ABC"
# LOCAL_DATA_DIR = os.path.join(LOCAL_EXTRACT_DIR, "MICCAI_BraTS2020_TrainingData_128")
LOCAL_DATA_DIR = LOCAL_EXTRACT_DIR

# Verify archive existence immediately before runtime allocation
assert os.path.exists(DATASET_ZIP), f"Dataset archive not found: {DATASET_ZIP}"

# Robust extraction guard: triggers if directory does not exist or is completely empty
# if not os.path.exists(LOCAL_EXTRACT_DIR) or len(os.listdir(LOCAL_EXTRACT_DIR)) == 0:
if not os.path.exists(LOCAL_DATA_DIR) or len(os.listdir(LOCAL_DATA_DIR)) == 0:
    print(f"Extracting preprocessed dataset to fast local runtime storage: {LOCAL_EXTRACT_DIR}...")
    os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)
    shutil.unpack_archive(DATASET_ZIP, LOCAL_EXTRACT_DIR, "zip")
    print("Extraction complete. Preprocessed dataset ready for I/O operations.")
else:
    print("Valid local preprocessed dataset cache detected. Skipping extraction.")

Extracting preprocessed dataset to fast local runtime storage: /content/MICCAI_BraTS2020_TrainingData_ABC...
Extraction complete. Preprocessed dataset ready for I/O operations.


# Component Integration and Framework Imports

In [3]:
!pip install -q monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.3 MB/s eta 0:00:00


In [4]:
import sys
import random
import numpy as np
import torch
import torch.optim as optim
import itertools

# Enforce strict scientific reproducibility thresholds across packages
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Optimize CUDA runtime convolution algorithm selection for static tensor patches
torch.backends.cudnn.benchmark = True

# Add src to path
PROJECT_ROOT = "/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor"
sys.path.append(PROJECT_ROOT)

import src.config as config
from src.dataset import get_brats_dataloaders
from src.losses import SegmentationLoss
from src.engine import run_training

# Instantiate the 5 distinct sub-network modules forming the end-to-end multi-task architecture
from src.models.mamba_backbone import MambaBackbone
from src.models.fusion import PresenceAwareCrossModalFusion
from src.models.mamba_backbone import SharedDeepMambaBackbone
from src.models.decoder import SegmentationDecoder3D, AuxiliaryDecoder3D

# Execution

In [5]:
import importlib

importlib.reload(config)

import src.engine as engine
importlib.reload(engine)

import src.models.mamba_backbone as mbb
importlib.reload(mbb)

import src.models.fusion as mf
importlib.reload(mf)

import src.models.decoder as md
importlib.reload(md)

import src.losses as sl
importlib.reload(sl)

import src.transforms as tf
importlib.reload(tf)

<module 'src.transforms' from '/content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/src/transforms.py'>

In [6]:
from torch.amp import GradScaler

# Hardware runtime verification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Operational Hardware target identified: {device}")
print(f"Target Checkpoint Saving Directory: {config.CHECKPOINT_DIR}")

# 1. Pipeline Dataset Loaders Construction
print("Instantiating MONAI dictionary data pipelines...")
train_loader, val_loader = get_brats_dataloaders()

# 2. Structural Module Instantiations
print("Initializing neural net components...")
backbone = MambaBackbone(embed_dim=config.EMBED_DIM).to(device)
fusion = PresenceAwareCrossModalFusion(embed_dim=config.EMBED_DIM).to(device)
shared_backbone = SharedDeepMambaBackbone(embed_dim=config.EMBED_DIM).to(device)
decoder = SegmentationDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)
aux_decoder = AuxiliaryDecoder3D(embed_dim=config.EMBED_DIM, out_channels=config.NUM_SEG_CLASSES).to(device)

model_components = (backbone, fusion, shared_backbone, decoder, aux_decoder)

# Compute total parameter profile summary metrics for publication tracking
total_params = sum(p.numel() for model in model_components for p in model.parameters())
print(f"Total Multi-Task Trainable Network Parameters: {total_params:,}")

# 3. Unified Multi-Task Optimization System Setup
criterion = SegmentationLoss().to(device)

# Chain layer parameters
all_parameters = itertools.chain(
    backbone.parameters(),
    fusion.parameters(),
    shared_backbone.parameters(),
    decoder.parameters(),
    aux_decoder.parameters()
)

optimizer = optim.AdamW(
    all_parameters,
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Automatically syncs decay frequency to match relative incremental epochs run window
# scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
#     optimizer,
#     # T_0=config.NUM_EPOCHS,
#     T_0=config.TOTAL_EPOCHS, # Currently set to 150
#     T_mult=1,
#     eta_min=config.ETA_MIN
# )

scheduler = optim.lr_scheduler.PolynomialLR(
    optimizer,
    total_iters=config.TOTAL_EPOCHS,
    power=0.9
)

# Device-aware mixed-precision gradient scaling framework
scaler = GradScaler(enabled=(device.type == "cuda"))

# 4. Trigger Orchestration Engine Pipeline
print("Handing execution loop management over to stateful runtime engine...")
run_training(
    model_components=model_components,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    device=device
)

Operational Hardware target identified: cuda
Target Checkpoint Saving Directory: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints
Instantiating MONAI dictionary data pipelines...
Initializing neural net components...
Total Multi-Task Trainable Network Parameters: 5,878,986
Handing execution loop management over to stateful runtime engine...
[*] Found existing checkpoint record at: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/latest_checkpoint.pth. Loading state...
[+] Recovery complete. Resuming from absolute internal epoch counter: 280
[*] Incremental Run Configuration: Training from Epoch 280 -> Target Epoch 300 (+20 epochs)

--- Epoch 281/300 ---


[Train] Seg Loss: 1.4177


[Val] Segmentation Loss-> Mean Dice: 0.8585 (WT: 0.8989, TC: 0.8732, ET: 0.8033)
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/latest_checkpoint.pth

--- Epoch 282/300 ---


[Train] Seg Loss: 1.3452


[Val] Segmentation Loss-> Mean Dice: 0.8508 (WT: 0.8804, TC: 0.8690, ET: 0.8030)
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/latest_checkpoint.pth

--- Epoch 283/300 ---


...

[Val] Segmentation Loss-> Mean Dice: 0.8546 (WT: 0.8935, TC: 0.8674, ET: 0.8030)
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/latest_checkpoint.pth

--- Epoch 299/300 ---


[Train] Seg Loss: 1.3498


[Val] Segmentation Loss-> Mean Dice: 0.8547 (WT: 0.8939, TC: 0.8673, ET: 0.8029)
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/latest_checkpoint.pth

--- Epoch 300/300 ---


[Train] Seg Loss: 1.4129


[Val] Segmentation Loss-> Mean Dice: 0.8548 (WT: 0.8941, TC: 0.8670, ET: 0.8032)
Stateful tracking saved to: /content/drive/MyDrive/ML_Projects/3D_MRI_Brain_Tumor/checkpoints/latest_checkpoint.pth

 Incremental cycle finished successfully. Total absolute epochs processed: 300
